# Testova detektori — SINOV MODELI (xlm-roberta-base, o'zbekcha)

> **Bu sinov modeli.** Ma'lumot kichik (bir necha yuz insho) va bitta manbadan
> (bitta Telegram kanali, bir xil janr — insho). Natijalar boshqa manbalarga
> o'tkazib bo'lmaydi. Model botga ULANMAYDI: u hech bir haqiqiy til kodiga
> bog'lanmagan (`uz-sinov`) va servis o'qimaydigan `detector/models_sinov/`
> papkasiga qo'yiladi. O'zbekcha uchun botda `supported=false` qoladi.

**Runtime → Change runtime type → T4 GPU** (bepul).

## Kompyuterda, oldindan

```bash
cd "<repo>"
P=detector/.venv/Scripts/python
$P -m detector.training.transcribe_telegram transcribe --env-file "<repo>/.env"
$P -m detector.training.transcribe_telegram build
$P -m detector.training.transcribe_telegram review      # ko'z bilan tekshiring
$P -m detector.training.prepare_human
$P -m detector.training.generate_ai generate --env-file "<repo>/.env"   # yoki DETECTOR_GEN_GEMINI_KEY muhitda
$P -m detector.training.generate_ai manual
$P -m detector.training.split
$P -m detector.training.augment_ocr
git archive HEAD detector/__init__.py detector/service detector/scripts -o detector_code.zip
```

Bu notebook'ga yuklang: `detector/data/work/split/train_aug.jsonl`,
`detector/data/work/split/val.jsonl`, `detector_code.zip`.

**`test.jsonl` bu yerga YUKLANMAYDI.** Test faqat kompyuterda, eksport qilingan
ONNX modelda o'lchanadi. Eslatma: Telegram insholari hammasi "tasdiqlanmagan"
(2022-11 dan keyin), shuning uchun test to'plami bo'sh bo'lishi mumkin — u holda
faqat val natijasi bor, va bu ham ishonchli emas.

In [ ]:
!pip -q install "transformers>=4.45,<6" datasets accelerate onnx onnxruntime safetensors scikit-learn

In [ ]:
from google.colab import files
up = files.upload()   # train_aug.jsonl, val.jsonl, detector_code.zip
assert "test.jsonl" not in up, "test.jsonl must not be uploaded"
print(list(up))

In [ ]:
import json, collections, numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import roc_auc_score

set_seed(13)
BASE = "xlm-roberta-base"
MAX_LEN = 512

def load(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    print(path, len(rows), collections.Counter(r["label"] for r in rows),
          collections.Counter(r.get("source", "?").split(":")[0] for r in rows))
    return Dataset.from_list([{"id": r["id"], "text": r["text"], "label": int(r["label"])} for r in rows])

train_ds, val_ds = load("train_aug.jsonl"), load("val.jsonl")
assert len(set(train_ds["label"])) == 2 and len(set(val_ds["label"])) == 2, "both classes needed"

tok = AutoTokenizer.from_pretrained(BASE)
enc = lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN)
train_tok = train_ds.map(enc, batched=True, remove_columns=["text", "id"])
val_tok = val_ds.map(enc, batched=True, remove_columns=["text", "id"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=2, id2label={0: "human", 1: "ai"}, label2id={"human": 0, "ai": 1})

def metrics(p):
    logits, labels = p
    prob = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
    human = np.sort(prob[labels == 0])[::-1]
    t = human[int(0.05 * len(human))] + 1e-6 if len(human) else 0.5   # FPR <= 5%
    tpr = float((prob[labels == 1] >= t).mean()) if (labels == 1).any() else float("nan")
    return {"auroc": roc_auc_score(labels, prob), "tpr_at_fpr5": tpr,
            "acc_at_0.5": float(((prob >= 0.5) == (labels == 1)).mean())}

# Small data: a few more epochs, small batches, eval every epoch, keep the best.
args = TrainingArguments(
    output_dir="ckpt", num_train_epochs=5, learning_rate=2e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    warmup_ratio=0.1, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="auroc",
    logging_steps=10, report_to="none", seed=13)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

In [ ]:
CKPT = "/content/uz-sinov-ckpt"
trainer.save_model(CKPT); tok.save_pretrained(CKPT)
!rm -rf code && mkdir code && cd code && unzip -q ../detector_code.zip
# "uz-sinov" is not a language the service ever detects: even if this folder
# were dropped into detector/models/, no real text would be routed to it.
!cd code && python -m detector.scripts.export_onnx --model-id {CKPT} --out /content/uz-sinov \
    --languages uz-sinov --name uz-xlmr-sinov --version uz-xlmr-sinov-v0-int8 --license MIT

In [ ]:
!cd /content && zip -qr uz-sinov.zip uz-sinov && ls -la uz-sinov.zip
files.download("/content/uz-sinov.zip")

## Kompyuterda, keyin

```bash
cd "<repo>"
mkdir -p detector/models_sinov && unzip uz-sinov.zip -d detector/models_sinov/
detector/.venv/Scripts/python -m detector.training.evaluate --model-dir detector/models_sinov/uz-sinov
```

`evaluate` chegarani val'da tanlaydi (FPR ≤ 5%) va test bo'lsa test'da o'lchaydi;
natija `detector/data/work/eval/report.json` va `test_scores.csv`.
`--write-threshold` ni sinov modelida ishlatmang. Botga ulash — faqat siz
natijani ko'rib qaror qilganingizdan keyin (`languages` ni `uz` ga o'zgartirib,
papkani `detector/models/` ga ko'chirish).